# 01 - PDF Upload and Transcript Loading

## Purpose

This notebook starts Version 2 of the PDF RAG chatbot project.

Version 2 upgrades the original implementation by using OpenAI models and a more structured LangChain workflow.

The goal is to load an uploaded PDF document from Databricks storage, extract the page-level text, and prepare a full transcript that can later be split, embedded, stored in a vector database, and used for Q&A.

## Main Changes in Version 2

Compared with the first implementation, this version will use:

- OpenAI embeddings instead of Hugging Face embeddings
- Chroma instead of FAISS
- Token-based splitting instead of only character-based splitting
- LangChain prompt templates instead of simple Python prompt strings
- LCEL-style chaining instead of only manual function calls
- GPT-based answer generation instead of FLAN-T5
- Streaming responses for a more chatbot-like experience

## Input

Uploaded PDF path:

```text
/Volumes/workspace/365pdf/365pdfupload/Introduction_to_Tableau.pdf

In [0]:
%pip install langchain langchain-community langchain-openai pypdf python-dotenv tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
import os

os.environ["OPENAI_API_KEY"] = "Place_Key_Here"

In [0]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "/Volumes/workspace/365pdf/365pdfupload/Introduction_to_Tableau.pdf"

loader_pdf = PyPDFLoader(pdf_path)

docs_list = loader_pdf.load()

print("PDF loaded successfully")
print("Number of pages loaded:", len(docs_list))
print()
print("First document metadata:")
print(docs_list[0].metadata)
print()
print("First page preview:")
print(docs_list[0].page_content[:1000])

In [0]:
string_list_concat = ""

for doc in docs_list:
    page_number = doc.metadata.get("page", "unknown")
    page_text = doc.page_content

    string_list_concat += f"\n\n--- Page {page_number} ---\n"
    string_list_concat += page_text

print("Combined transcript created")
print("Total characters:", len(string_list_concat))
print()
print(string_list_concat[:2000])

In [0]:
from pyspark.sql import Row

raw_rows = []

for i, doc in enumerate(docs_list):
    raw_rows.append(
        Row(
            page_id=i,
            source=str(doc.metadata.get("source")),
            page_number=int(doc.metadata.get("page", i)),
            text=str(doc.page_content)
        )
    )

print("Rows prepared:", len(raw_rows))
print("First row preview:")
print(raw_rows[0].text[:500])

In [0]:
print("Number of loaded PDF pages:", len(docs_list))

for i, doc in enumerate(docs_list[:3]):
    print(f"\nPage document {i}")
    print("Metadata:", doc.metadata)
    print("Preview:")
    print(doc.page_content[:500])
    print("-" * 80)

In [0]:
print("Transcript variable created")
print("Total characters:", len(string_list_concat))
print(string_list_concat[:1000])